## 11 Composite Analysis

**笔记本**：`11 composite_analysis20260403.1.ipynb`

**库**：`geopandas`、`pandas`、`numpy`、`matplotlib`、`shapely`、`scipy.spatial.cKDTree`、（可选）`h3`（`SPACING_MODE="h3_grid"` 时）等。

**输入**：
- `../04 Transport/data_raw/shenzhen_boundary.geojson`
- （格网定义）`../03 Boundary/data_out/sz_hex_grid_res8.gpkg` 由 03b 生成；主流程以 08 `sz_demand_grid.gpkg` 的 `h3_id` 为准与之对齐
- `../08 POI Demand/data_out/sz_demand_grid.gpkg`
- `../09 Population/data_out/sz_population_grid.gpkg`
- `../10 OD & Ground Friction/data_out/sz_od_analysis.gpkg`、`sz_routes.gpkg`
- `../07 Parks & Compounds/data_out/sz_compounds.gpkg`
- `../06 Buildings/data_out/sz_building_grid.gpkg`
- `../05 Barrier Layers/data_out/sz_barrier_unions.gpkg`
- （可选）`../10 OD & Ground Friction/data_out/sz_waimai_context_grid.gpkg`

**做了什么 / 算了什么**：
- 在统一 H3 res=8 网格（h3_id，来自 08 demand）上合并 demand_pressure、intensity_index / pop_count、由 OD 聚合的 avg_friction / avg_detour / avg_congestion 等。
- gap_index：demand_norm × friction_norm、intensity × friction、demand × intensity 三项加权（0.4 / 0.3 / 0.3）。
当前地面系统最可能“供给不足”或“值得无人机补位”的空间。
- 候选起降点：高 gap、避开水体、最小间距约900m，TOP_N=200；最近 compound 类型与距离（让站点推荐更有“城市语义”，而不是抽象数学点。）；按 gap 分 hub / station / endpoint。
- 策略模拟：对 top-K 站点做 3 km（以度表示）缓冲，用 cKDTree 算覆盖格及 demand_pressure、pop_count 覆盖比例（+3、+5、+10、+20、+50）。
为什么用 3 km buffer？这相当于定义一个简化的服务半径。
虽然它不是真实飞行航线模拟，但可以作为：站点影响范围 / 服务覆盖范围 proxy
- relief_vulnerability：以 +10 策略覆盖为基准，gap_index × (未覆盖)。

**写出文件**：
- `data_out/sz_gap_grid.gpkg`
- `data_out/sz_candidate_sites.gpkg`
- `data_out/sz_strategy_coverage.gpkg`
- `data_out/sz_strategy_{n}_sites.gpkg`（按策略导出的站点子集，若 notebook 中启用）

**典型输出信息**：各上游表行数、`Unified grid` 格数、合并字段提示、Gap 统计与 Top 格、`Candidate sites` / `Strategy maps` 保存路径。
-1. 3 km buffer 是简化服务范围；它不是严格飞行走廊、法规约束下的真实可达域。
-2. gap_index 是 proxy，不是真实运营收益；它表达补位潜力，而不是财务回报或真实订单增长。
-3. 候选站点是规则筛选结果；不等于现实中立即可建设的合法起降设施。
-4. 未覆盖不等于绝对不可服务；只是说在当前策略半径和点位设置下没被纳入。


In [ ]:
# ============================================================
#  11 Composite Analysis
#  汇聚所有上游数据 → demand × friction overlap → 选址 → 策略
#  输入: 08 POI + 09 Population + 10 OD/Friction + 10b 公服上下文(可选) + 07 Compounds
#  输出: gap index / candidate sites / strategy maps
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
from shapely.ops import unary_union
from shapely.prepared import prep as shapely_prep
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings("ignore")

OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

# ── 上游数据路径 ──
BOUNDARY     = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")
OD_ANALYSIS  = Path("../10 OD & Ground Friction/data_out/sz_od_analysis.gpkg")
ROUTES       = Path("../10 OD & Ground Friction/data_out/sz_routes.gpkg")
DEMAND_GRID  = Path("../08 POI Demand/data_out/sz_demand_grid.gpkg")
POP_GRID     = Path("../09 Population/data_out/sz_population_grid.gpkg")
COMPOUNDS    = Path("../07 Parks & Compounds/data_out/sz_compounds.gpkg")
BLDG_GRID    = Path("../06 Buildings/data_out/sz_building_grid.gpkg")
BARRIERS     = Path("../05 Barrier Layers/data_out/sz_barrier_unions.gpkg")
WAIMAI_CTX   = Path("../10 OD & Ground Friction/data_out/sz_waimai_context_grid.gpkg")

# ── 加载 ──
shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union

print("Loading upstream data ...")
od = gpd.read_file(OD_ANALYSIS) if OD_ANALYSIS.exists() else None
demand = gpd.read_file(DEMAND_GRID) if DEMAND_GRID.exists() else None
pop = gpd.read_file(POP_GRID) if POP_GRID.exists() else None
compounds = gpd.read_file(COMPOUNDS) if COMPOUNDS.exists() else None
bldg_grid = gpd.read_file(BLDG_GRID) if BLDG_GRID.exists() else None
waimai_ctx = gpd.read_file(WAIMAI_CTX) if WAIMAI_CTX.exists() else None

for name, df in [("OD", od), ("Demand", demand), ("Pop", pop),
                  ("Compounds", compounds), ("BldgGrid", bldg_grid),
                  ("WaimaiCtx", waimai_ctx)]:
    if df is not None:
        print(f"  {name:12s} {len(df):>8,} rows")
    else:
        print(f"  {name:12s} NOT FOUND")

Loading upstream data ...
  OD             76,177 rows
  Demand          (H3 res8 格数，重新运行后更新)
  Pop             (同左)
  Compounds      19,637 rows
  BldgGrid        (同 Demand)


In [2]:
# ============================================================
#  1. Demand × Friction Overlap → Gap Index
#  统一网格 = 08 demand（H3 res=8, h3_id）
# ============================================================

if demand is None:
    raise FileNotFoundError("需要 08 的 sz_demand_grid.gpkg（含 h3_id）")

grid = demand.copy()
grid["cx"] = grid.geometry.centroid.x
grid["cy"] = grid.geometry.centroid.y
print(f"Unified grid: {len(grid):,} H3 cells (from 08)")

# ── demand 列已在 grid 上；缺省时填 0 ──
if "demand_pressure" not in grid.columns:
    grid["demand_pressure"] = 0
    print("  demand_pressure: NOT AVAILABLE")
else:
    print("  demand_pressure: from 08 grid")

# ── 合并 intensity_index (from 09) ──
if pop is not None and "intensity_index" in pop.columns:
    pop_sub = pop[["h3_id", "intensity_index", "pop_count"]].copy()
    grid = grid.merge(pop_sub, on="h3_id", how="left")
    grid["intensity_index"] = grid["intensity_index"].fillna(0)
    grid["pop_count"] = grid["pop_count"].fillna(0)
    print(f"  intensity_index merged")
else:
    grid["intensity_index"] = 0
    grid["pop_count"] = 0
    print("  intensity_index: NOT AVAILABLE")

# ── 合并 ground_friction (from 10, 聚合到六边形) ──
if od is not None and "ground_friction" in od.columns:
    od_pts_o = gpd.GeoDataFrame(
        od[["ground_friction", "detour_ratio", "congestion_amplifier"]],
        geometry=gpd.points_from_xy(od["o_lon"], od["o_lat"]),
        crs=4326,
    )
    oj = gpd.sjoin(od_pts_o, grid[["h3_id", "geometry"]], how="inner", predicate="within")
    friction_by_grid = oj.groupby("h3_id").agg(
        avg_friction=("ground_friction", "mean"),
        avg_detour=("detour_ratio", "mean"),
        avg_congestion=("congestion_amplifier", "mean"),
    ).reset_index()
    grid = grid.merge(friction_by_grid, on="h3_id", how="left")
    grid[["avg_friction", "avg_detour", "avg_congestion"]] = grid[
        ["avg_friction", "avg_detour", "avg_congestion"]
    ].fillna(0)
    print(f"  ground_friction merged (from {len(friction_by_grid)} cells)")
else:
    grid["avg_friction"] = 0
    grid["avg_detour"] = 0
    grid["avg_congestion"] = 0
    print("  ground_friction: NOT AVAILABLE")

# ── 合并 10b（可选）──
if waimai_ctx is not None and "h3_id" in waimai_ctx.columns:
    extra = waimai_ctx.drop(columns=[c for c in waimai_ctx.columns if c == "geometry"])
    grid = grid.merge(extra, on="h3_id", how="left")
    wcols = [c for c in extra.columns if c != "h3_id"]
    grid[wcols] = grid[wcols].fillna(0)
    print(f"  waimai_context merged ({len(wcols)} cols from 10b)")
else:
    print("  waimai_context: NOT AVAILABLE (run 10b_waimai_facilities_context.ipynb)")

# ── Gap Index ──
def norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else s * 0

grid["demand_norm"] = norm(grid["demand_pressure"])
grid["friction_norm"] = norm(grid["avg_friction"])
grid["intensity_norm"] = norm(grid["intensity_index"])

grid["gap_index"] = (
    0.4 * grid["demand_norm"] * grid["friction_norm"]
    + 0.3 * grid["intensity_norm"] * grid["friction_norm"]
    + 0.3 * grid["demand_norm"] * grid["intensity_norm"]
)

non_zero = grid["gap_index"] > 0
print(f"\n=== Gap Index ===")
print(f"Grids with gap > 0: {non_zero.sum():,} / {len(grid):,}")
print(
    f"Gap: mean={grid.loc[non_zero, 'gap_index'].mean():.4f}  max={grid['gap_index'].max():.4f}"
)
print(f"\nTop 10 gap cells:")
print(
    grid.nlargest(10, "gap_index")[
        ["h3_id", "demand_pressure", "avg_friction", "intensity_index", "gap_index"]
    ].to_string(index=False)
)


Unified grid: (H3 res8，重新运行后更新格数)
  demand_pressure merged
  intensity_index merged
  ground_friction merged (from 393 grids)

=== Gap Index ===
Grids with gap > 0: 4,801 / 7,392
Gap: mean=0.0089  max=0.5350

Top 10 gap grids:
 h3_id  demand_pressure  avg_friction  intensity_index  gap_index
 (重新运行后显示 H3 单元格 ID)


In [3]:
# ============================================================
#  2. Candidate Site Pool (候选起降点)
#  筛选逻辑:
#    - gap_index 高的网格中心 (需求大 + 地面难)
#    - 不在水体/铁路 barrier 内
#    - 靠近 compound (有屋顶/空地可用)
#    - 按 gap_index 排名, 保留 top N
# ============================================================

# 排除 barrier 区域
barrier_unions = gpd.read_file(BARRIERS) if BARRIERS.exists() else None
exclude_geom = None
if barrier_unions is not None:
    water_geom = barrier_unions[barrier_unions["barrier_type"] == "water"].geometry
    if len(water_geom) > 0:
        exclude_geom = unary_union(water_geom)
        exclude_prep = shapely_prep(exclude_geom)

# 从高 gap 网格中选候选点
TOP_N = 200
candidates = grid[grid["gap_index"] > 0].nlargest(TOP_N * 3, "gap_index").copy()
candidates["candidate_point"] = candidates.apply(
    lambda r: Point(r["cx"], r["cy"]), axis=1
)

# 过滤: 排除落在水体内的
if exclude_geom is not None:
    candidates["in_water"] = candidates["candidate_point"].apply(lambda pt: exclude_prep.contains(pt))
    candidates = candidates[~candidates["in_water"]].drop(columns=["in_water"])

# 过滤: 与已选候选点保持最小间距 (避免扎堆)
#  SPACING_MODE:
#    "haversine_m" — 大圆距离 ≥ MIN_SPACING_M（米），各向同性
#    "h3_grid"     — 同分辨率 H3 grid_distance ≥ MIN_H3_GRID_STEPS
from math import radians, sin, cos, sqrt, atan2

def _haversine_m(lon1, lat1, lon2, lat2) -> float:
    R = 6_371_000.0
    la1, la2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlmb = radians(lon2 - lon1)
    a = sin(dphi / 2) ** 2 + cos(la1) * cos(la2) * sin(dlmb / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(max(0.0, 1.0 - a)))


SPACING_MODE = "haversine_m"  # "haversine_m" | "h3_grid"
MIN_SPACING_M = 900.0
MIN_H3_GRID_STEPS = 2  # h3_grid: 1=仅排除直接相邻格

selected = []
selected_coords = []
selected_h3 = []
if SPACING_MODE == "h3_grid":
    import h3

for _, row in candidates.iterrows():
    pt = (float(row["cx"]), float(row["cy"]))
    too_close = False
    if SPACING_MODE == "haversine_m":
        for sc in selected_coords:
            if _haversine_m(pt[0], pt[1], sc[0], sc[1]) < MIN_SPACING_M:
                too_close = True
                break
    else:
        hid = str(row["h3_id"])
        for sh in selected_h3:
            try:
                d = h3.grid_distance(hid, sh)
            except Exception:
                d = 0
            if d < MIN_H3_GRID_STEPS:
                too_close = True
                break
    if not too_close:
        selected.append(row)
        selected_coords.append(pt)
        if SPACING_MODE == "h3_grid":
            selected_h3.append(str(row["h3_id"]))
    if len(selected) >= TOP_N:
        break

site_pool = gpd.GeoDataFrame(selected, crs=4326)
site_pool["geometry"] = site_pool["candidate_point"]
site_pool = site_pool.drop(columns=["candidate_point"]).reset_index(drop=True)
site_pool["site_id"] = range(len(site_pool))

# 为每个候选点标注最近 compound 类型
if compounds is not None:
    comp_centroids = np.array(list(zip(
        compounds.geometry.centroid.x, compounds.geometry.centroid.y
    )))
    comp_tree = cKDTree(comp_centroids)
    dists, idxs = comp_tree.query(np.array(list(zip(site_pool["cx"], site_pool["cy"]))))
    site_pool["nearest_compound_type"] = compounds.iloc[idxs]["compound_type"].values
    site_pool["nearest_compound_dist_m"] = dists * 111320

# 分类: hub / station / endpoint
site_pool["site_class"] = pd.cut(
    site_pool["gap_index"],
    bins=[0, site_pool["gap_index"].quantile(0.33),
          site_pool["gap_index"].quantile(0.66), 1],
    labels=["endpoint", "station", "hub"],
)

print(f"Candidate sites: {len(site_pool)}")
print(f"\nBy class:")
print(site_pool["site_class"].value_counts().to_string())
print(f"\nBy nearest compound:")
print(site_pool["nearest_compound_type"].value_counts().to_string())

Candidate sites: 200

By class:
site_class
hub         68
endpoint    66
station     66

By nearest compound:
nearest_compound_type
residential    110
industrial      30
commercial      24
campus          20
park            16


In [4]:
# ============================================================
#  3. +3 / +5 / +10 Strategy Simulation
#  从候选池中选 top-K 个站点, 计算覆盖范围和收益
#  覆盖 = 深圳附近 EPSG:32650 下半径 COVERAGE_RADIUS_M（米）的圆盘并集
# ============================================================

PROJ_M = "EPSG:32650"
COVERAGE_RADIUS_M = 3000.0
STRATEGIES = [3, 5, 10]

_grid_pts = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(grid["cx"], grid["cy"], crs="EPSG:4326"),
    crs="EPSG:4326",
).to_crs(PROJ_M)
grid_coords = np.column_stack([_grid_pts.geometry.x, _grid_pts.geometry.y])
grid_tree = cKDTree(grid_coords)

def evaluate_strategy(sites_gdf, n_sites, grid_df, grid_tree, radius_m):
    """选 top-n 站点；KDTree 与 buffer 均在 PROJ_M 米制下。"""
    top = sites_gdf.head(n_sites)
    top_m = top.to_crs(PROJ_M)

    covered_grids = set()
    for _, site in top_m.iterrows():
        g = site.geometry
        idxs = grid_tree.query_ball_point([g.x, g.y], radius_m)
        covered_grids.update(idxs)

    total_demand = grid_df["demand_pressure"].sum()
    covered_demand = grid_df.iloc[list(covered_grids)]["demand_pressure"].sum()
    total_pop = grid_df["pop_count"].sum()
    covered_pop = grid_df.iloc[list(covered_grids)]["pop_count"].sum()

    coverage_area_pct = len(covered_grids) / len(grid_df) * 100
    demand_coverage_pct = covered_demand / total_demand * 100 if total_demand > 0 else 0
    pop_coverage_pct = covered_pop / total_pop * 100 if total_pop > 0 else 0

    buffers = top_m.geometry.buffer(radius_m)
    coverage_union_m = unary_union(list(buffers))
    coverage_union = gpd.GeoSeries([coverage_union_m], crs=PROJ_M).to_crs(4326).iloc[0]

    return {
        "n_sites": n_sites,
        "covered_grids": len(covered_grids),
        "coverage_area_pct": coverage_area_pct,
        "demand_coverage_pct": demand_coverage_pct,
        "pop_coverage_pct": pop_coverage_pct,
        "covered_demand": covered_demand,
        "covered_pop": covered_pop,
        "site_ids": list(top["site_id"]),
        "coverage_geom": coverage_union,
    }

# 按 gap_index 排序 (hub 优先)
site_pool_sorted = site_pool.sort_values("gap_index", ascending=False).reset_index(drop=True)

strategy_results = []
print(f"=== Strategy Simulation ({COVERAGE_RADIUS_M/1000:.0f} km metric, {PROJ_M}) ===\n")
print(f"{'Strategy':>10s} {'Sites':>6s} {'Area%':>7s} {'Demand%':>8s} {'Pop%':>7s}")
print("-" * 42)

for n in STRATEGIES:
    r = evaluate_strategy(site_pool_sorted, n, grid, grid_tree, COVERAGE_RADIUS_M)
    strategy_results.append(r)
    print(f"  +{n:<8d} {r['n_sites']:>6d} {r['coverage_area_pct']:>6.1f}% {r['demand_coverage_pct']:>7.1f}% {r['pop_coverage_pct']:>6.1f}%")

# 也算 +20, +50 看边际收益
for n in [20, 50]:
    if n <= len(site_pool_sorted):
        r = evaluate_strategy(site_pool_sorted, n, grid, grid_tree, COVERAGE_RADIUS_M)
        strategy_results.append(r)
        print(f"  +{n:<8d} {r['n_sites']:>6d} {r['coverage_area_pct']:>6.1f}% {r['demand_coverage_pct']:>7.1f}% {r['pop_coverage_pct']:>6.1f}%")

=== Strategy Simulation (3km coverage) ===

  Strategy  Sites   Area%  Demand%    Pop%
------------------------------------------
  +3             3    3.7%     9.5%    8.0%
  +5             5    6.0%    18.7%   15.9%
  +10           10   10.7%    31.0%   27.8%
  +20           20   18.3%    46.2%   41.9%
  +50           50   35.7%    75.8%   69.7%


In [5]:
# ============================================================
#  4. Relief Vulnerability Map + 保存所有输出
#  relief_vulnerability = gap_index × (1 - coverage)
#  未被任何策略覆盖 + gap 高 → 最需要被"救济"的区域
# ============================================================

# ── Relief Vulnerability: 用 +10 策略作为基准 ──
best_strategy = [r for r in strategy_results if r["n_sites"] == 10]
if best_strategy:
    coverage_geom = best_strategy[0]["coverage_geom"]
    coverage_prep = shapely_prep(coverage_geom)

    grid["covered_by_10"] = grid.apply(
        lambda r: coverage_prep.intersects(Point(r["cx"], r["cy"])), axis=1
    )
    grid["relief_vulnerability"] = grid["gap_index"] * (~grid["covered_by_10"]).astype(float)

    uncovered_high = grid[(~grid["covered_by_10"]) & (grid["gap_index"] > grid["gap_index"].quantile(0.75))]
    print(f"=== Relief Vulnerability (+10 baseline) ===")
    print(f"Covered grids: {grid['covered_by_10'].sum():,} / {len(grid):,}")
    print(f"Uncovered + high gap: {len(uncovered_high):,} grids")
    print(f"Max vulnerability: {grid['relief_vulnerability'].max():.4f}")
else:
    grid["covered_by_10"] = False
    grid["relief_vulnerability"] = grid["gap_index"]
    print("No +10 strategy available")

# ══════════════════════════════════════════
#  保存所有输出
# ══════════════════════════════════════════

# 1) Gap Index 网格
grid.to_file(OUT / "sz_gap_grid.gpkg", driver="GPKG")
print(f"\nGap grid       -> data_out/sz_gap_grid.gpkg  ({len(grid):,} cells)")

# 2) 候选站点
site_pool.to_file(OUT / "sz_candidate_sites.gpkg", driver="GPKG")
print(f"Candidate sites -> data_out/sz_candidate_sites.gpkg  ({len(site_pool):,} sites)")

# 3) 策略覆盖圈
strategy_rows = []
for r in strategy_results:
    strategy_rows.append({
        "n_sites": r["n_sites"],
        "coverage_area_pct": round(r["coverage_area_pct"], 1),
        "demand_coverage_pct": round(r["demand_coverage_pct"], 1),
        "pop_coverage_pct": round(r["pop_coverage_pct"], 1),
        "geometry": r["coverage_geom"],
    })
strategy_gdf = gpd.GeoDataFrame(strategy_rows, crs=4326)
strategy_gdf.to_file(OUT / "sz_strategy_coverage.gpkg", driver="GPKG")
print(f"Strategy maps   -> data_out/sz_strategy_coverage.gpkg  ({len(strategy_gdf)} strategies)")

# 4) Top sites per strategy
for r in strategy_results:
    n = r["n_sites"]
    top_sites = site_pool[site_pool["site_id"].isin(r["site_ids"])]
    top_sites.to_file(OUT / f"sz_strategy_{n}_sites.gpkg", driver="GPKG")

print("\n=== Summary ===")
print(f"Gap index range: {grid['gap_index'].min():.4f} - {grid['gap_index'].max():.4f}")
print(f"Candidate sites: {len(site_pool)} (hub: {(site_pool['site_class']=='hub').sum()}, station: {(site_pool['site_class']=='station').sum()}, endpoint: {(site_pool['site_class']=='endpoint').sum()})")
print(f"\nStrategy comparison:")
for r in strategy_results:
    print(f"  +{r['n_sites']}: {r['demand_coverage_pct']:.1f}% demand covered, {r['pop_coverage_pct']:.1f}% population covered")

=== Relief Vulnerability (+10 baseline) ===
Covered grids: 794 / 7,392
Uncovered + high gap: 1,368 grids
Max vulnerability: 0.2434

Gap grid       -> data_out/sz_gap_grid.gpkg  (7,392 cells)
Candidate sites -> data_out/sz_candidate_sites.gpkg  (200 sites)
Strategy maps   -> data_out/sz_strategy_coverage.gpkg  (5 strategies)

=== Summary ===
Gap index range: 0.0000 - 0.5350
Candidate sites: 200 (hub: 68, station: 66, endpoint: 66)

Strategy comparison:
  +3: 9.5% demand covered, 8.0% population covered
  +5: 18.7% demand covered, 15.9% population covered
  +10: 31.0% demand covered, 27.8% population covered
  +20: 46.2% demand covered, 41.9% population covered
  +50: 75.8% demand covered, 69.7% population covered
